# SiteSafe v2 on Kaggle (free GPU)

Same run as the Colab v2 notebook: five real-photo datasets, uncertain regions painted grey so they
never count as background, a fine-tune of the current app model, then a ship / don't-ship verdict
on the test split and my 30 own photos. About 2.5 hours on a T4.

**One-time setup**
1. kaggle.com → Settings → verify your phone number (needed for GPU and internet).
2. Datasets → New dataset → upload `sitesafe_v2_code.zip` (keep it as a zip), any title, Create.
3. Code → New notebook → File → Import notebook → this file.
4. Right-hand panel: **Add input** → your dataset; **Accelerator → GPU T4 x2** (not P100: the
   current PyTorch no longer supports it); **Internet → On**.

**Run it in the background (browser can be closed)**
Save Version (top right) → **Save & Run All (Commit)** → Save. Close the tab if you like.
When it finishes (Kaggle e-mails you), open the notebook → **Output** → download
`sitesafe_v2_results.zip`.

In [1]:
# 1. Settings
EPOCHS = 40
PATIENCE = 12
BATCH = 32

In [2]:
# 2. GPU, code bundle, packages
import os, subprocess, shutil, zipfile, time, json, glob
import torch
assert torch.cuda.is_available(), "No GPU: right-hand panel > Accelerator > GPU T4 x2"
print("GPU:", torch.cuda.get_device_name(0))
found = glob.glob("/kaggle/input/**/sitesafe_v2_code.zip", recursive=True)
W = "/tmp/sitesafe_v2"            # big data stays out of the saved output
os.makedirs(W, exist_ok=True)
if found:
    with zipfile.ZipFile(found[0]) as z:
        z.extractall(W)
else:  # Kaggle sometimes unpacks uploaded zips: copy the folder instead
    marker = glob.glob("/kaggle/input/**/scripts/prepare_v2_data.py", recursive=True)
    assert marker, "Add the dataset with sitesafe_v2_code.zip as an input (right-hand panel > Add input)"
    shutil.copytree(os.path.dirname(os.path.dirname(marker[0])), W, dirs_exist_ok=True)
os.chdir(W)
OUT = "/kaggle/working"
os.makedirs(f"{OUT}/runs_v2", exist_ok=True)  # checkpoints are written straight into the saved output
if not os.path.islink("runs"):
    shutil.rmtree("runs", ignore_errors=True); os.symlink(f"{OUT}/runs_v2", "runs")
subprocess.run("pip -q install ultralytics==8.4.215 onnx onnxruntime onnxslim huggingface_hub", shell=True, check=True)
def sh(cmd):
    print(">", cmd, flush=True); t = time.time()
    r = subprocess.run(cmd, shell=True)
    print(f"  ({(time.time()-t)/60:.1f} min)", flush=True)
    if r.returncode: raise RuntimeError(f"failed: {cmd}")

GPU: Tesla T4


In [3]:
# 3. Data: Construction-PPE (identical split) + five real-photo datasets
sh("python scripts/download_data.py")
before = {n: open(f"data/splits/{n}.txt").read() for n in ("train", "val", "test")}
sh("python scripts/make_split.py")
for n, txt in before.items():
    assert open(f"data/splits/{n}.txt").read() == txt, f"{n} split differs from the laptop split!"
sh("python scripts/prepare_v2_data.py")
print(open("data/extra_v2/stats.json").read())

> python scripts/download_data.py
downloading https://github.com/ultralytics/assets/releases/download/v0.0.0/construction-ppe.zip
  done (647.7 MB)
unzipping ...
dataset ready: 800 train, 200 val images at /tmp/sitesafe_v2/data/construction-ppe
already present: /tmp/sitesafe_v2/models/face_detection_yunet_2023mar.onnx
  (0.1 min)
> python scripts/make_split.py
{
  "seed": 42,
  "train": 700,
  "val": 200,
  "test": 100,
  "test_images_per_class": {
    "helmet": 79,
    "gloves": 68,
    "vest": 82,
    "boots": 72,
    "goggles": 56,
    "Person": 100,
    "no_helmet": 44,
    "no_goggle": 32,
    "no_gloves": 45,
    "no_boots": 35,
    "forklift": 26,
    "no_vest": 38
  }
}
  (0.0 min)
> python scripts/prepare_v2_data.py


Fetching 6178 files:  94%|█████████▍| 5822/6178 [02:04<00:04, 73.11it/s]HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/HB1204/PPE_Detection/resolve/4d3c001e39d071a5e0f6f9b8c962a8456afcbd58/train/labels/image946_jpg.rf.b0584d7226b100cd2b5da526d012b4e3.txt
Rate limited. Waiting 54.0s before retry [Retry 1/5].
HTTP Error 429 thrown while requesting GET https://huggingface.co/api/resolve-cache/datasets/HB1204/PPE_Detection/4d3c001e39d071a5e0f6f9b8c962a8456afcbd58/train%2Flabels%2Fimage946_jpg.rf.13b6ff290c8e8f74a4daaa10e71892ed.txt?%2Fdatasets%2FHB1204%2FPPE_Detection%2Fresolve%2F4d3c001e39d071a5e0f6f9b8c962a8456afcbd58%2Ftrain%2Flabels%2Fimage946_jpg.rf.13b6ff290c8e8f74a4daaa10e71892ed.txt=&etag=%22b6a6fb8dbf7c25a16561c997a7538d74255ed976%22
Rate limited. Waiting 54.0s before retry [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/HB1204/PPE_Detection/resolve/4d3c001e39d071a5e0f6f9b8c962a8456afcbd58/train/labels/image947_

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
hb1204 {"seen": 1275, "dropped_overlap": 4, "dropped_area": 0, "ignore_regions": 214, "ignore_skipped": 0, "human": {"no_vest": 731, "helmet": 1534, "Person": 2003, "goggles": 463, "vest": 1434, "gloves": 1277, "boots": 1369, "no_helmet": 445, "no_goggle": 377, "no_gloves": 498}, "pseudo": {"Person": 173, "no_boots": 4}, "kept": 1271}
vincent {"seen": 1297, "dropped_overlap": 6, "dropped_area": 0, "ignore_regions": 6596, "ignore_skipped": 88, "human": {"Person": 3050, "no_helmet": 364, "no_vest": 1018, "helmet": 2559, "vest": 1352}, "pseudo": {"no_gloves": 978, "no_goggle": 343, "boots": 434, "no_boots": 51, "Person": 297, "goggles": 168, "gloves": 240}, "kept": 1291}
kerem_cs {"seen": 397, "dropped

In [4]:
# 4. Fine-tune the current app model on all of it
if not os.path.exists("runs/v2/results.png"):
    resume = "--resume" if os.path.exists("runs/v2/weights/last.pt") else ""
    sh(f"python scripts/train.py --name v2 --data ppe_v2.yaml --model models/sitesafe_base.pt --epochs {EPOCHS} "
       f"--patience {PATIENCE} --batch {BATCH} --workers 4 --device 0 --cache disk {resume}")

> python scripts/train.py --name v2 --data ppe_v2.yaml --model models/sitesafe_base.pt --epochs 40 --patience 12 --batch 32 --workers 4 --device 0 --cache disk 
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=disk, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/tmp/sitesafe_v2/data/splits/ppe_v2.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_r

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/40      5.79G      1.397      1.092      1.349         52        640: 100% ━━━━━━━━━━━━ 411/411 4.3it/s 1:360.2sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 2.6it/s 1.6s0.7s
                   all        200       2961      0.816      0.636      0.706      0.467

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/40       5.8G      1.386      1.084      1.346         39        640: 100% ━━━━━━━━━━━━ 411/411 4.3it/s 1:360.2sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 3.1it/s 1.3s0.6s
                   all        200       2961      0.781      0.642      0.691      0.453

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/40       5.8G      1.385      1.083      1.345         51        640: 

In [5]:
# 5. Evaluate, verdict, results zip in the saved output
errors = []
def step(cmd):
    try: sh(cmd)
    except Exception as e: errors.append(str(e)); print("  !!", e)
step("python scripts/export_onnx.py --weights runs/v2/weights/best.pt --name sitesafe_v2")
step("python scripts/evaluate.py --weights runs/v2/weights/best.pt --split test --tag v2")
step("python scripts/calibrate.py --model models/sitesafe_v2.onnx --tag v2")
step("python scripts/compliance_eval.py --model models/sitesafe_v2.onnx --tag v2 --split test")
step("python scripts/own_eval.py --models models/sitesafe_base.onnx models/sitesafe_v2.onnx")

verdict = "not evaluated"
try:
    o = json.load(open("results/eval/own_eval.json"))
    b, v = o["sitesafe_base"], o["sitesafe_v2"]
    keys = [("helmet_M_shown_M", "missing helmets called"), ("unsure_share", "sent to check manually"),
            ("nc_shown_green", "violators shown green"), ("vest_M_shown_W", "no-vest shown as vest"),
            ("boots_W_shown_W", "boots recognised"), ("accuracy", "right, of decided")]
    print(f"{'own 30 photos':28s}{'current':>10s}{'v2':>10s}")
    for k, label in keys:
        print(f"{label:28s}{b[k]:>10.2f}{v[k]:>10.2f}")
    ok = (v["helmet_M_shown_M"] >= b["helmet_M_shown_M"] - 0.1 and v["unsure_share"] <= b["unsure_share"] + 0.03
          and v["nc_shown_green"] <= b["nc_shown_green"] + 0.06)
    verdict = "SHIP: v2 is at least as good on the checks that matter" if ok else "DON'T SHIP: v2 is worse on a key check"
except Exception as e:
    errors.append(f"verdict: {e}")
print("\nVERDICT:", verdict)

out = f"{OUT}/sitesafe_v2_results.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in ("models/sitesafe_v2.onnx", "data/extra_v2/stats.json", "results/runs.csv"):
        if os.path.exists(f): z.write(f)
    if os.path.exists("runs/v2/weights/best.pt"):
        z.write("runs/v2/weights/best.pt", "models/sitesafe_v2.pt")
    for dp, _, fs in os.walk("results"):
        for f in fs: z.write(os.path.join(dp, f))
    for f in ("results.csv", "results.png", "args.yaml"):
        if os.path.exists(f"runs/v2/{f}"): z.write(f"runs/v2/{f}", f"runs/v2/{f}")
    z.writestr("verdict.txt", verdict + "\n" + "\n".join(errors))
print(f"Saved {out} ({os.path.getsize(out)/1e6:.0f} MB): download it from the notebook's Output tab.")
if errors: print("Some steps failed (details in verdict.txt):", errors)

> python scripts/export_onnx.py --weights runs/v2/weights/best.pt --name sitesafe_v2
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO11n summary (fused): 101 layers, 2,599,724 parameters, 0 gradients, 6.5 GFLOPs

PyTorch: starting from '/tmp/sitesafe_v2/models/_sitesafe_v2.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 16, 8400) (5.2 MB)

ONNX: starting export with onnx 1.23.2 opset 17...
ONNX: slimming with onnxslim 0.1.99...
ONNX: export success ✅ 2.7s, saved as '/tmp/sitesafe_v2/models/_sitesafe_v2.onnx' (10.2 MB)

Export complete (3.2s)
Results saved to /tmp/sitesafe_v2/models/_sitesafe_v2.onnx
Predict:         yolo predict task=detect model=/tmp/sitesafe_v2/models/_sitesafe_v2.onnx imgsz=640

/usr/lib/python3.13/zipfile/__init__.py:1686: UserWarning: Duplicate name: 'results/runs.csv'
  return self._open_to_write(zinfo, force_zip64=force_zip64)
